<h3>PDF의 텍스트 층과 표</h3>
<p><strong>개념</strong> PDF는 화면 배치를 저장하는 형식입니다. 글자가 선택되는 디지털 PDF도 읽기 순서와 표의 행·열 관계가 보존되는지 따로 확인해야 합니다.</p>

In [1]:
%pip install -q "pypdf==6.18.0" "pdfplumber==0.11.10"


Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from pypdf import PdfReader
import pdfplumber

roots = [Path.cwd(), *Path.cwd().parents]
repos = [p for p in roots if (p / "labs/data/rag-corpus").is_dir()]
if not repos: raise FileNotFoundError("저장소 루트에서 노트북을 실행하세요")
repo = repos[0]
pdf_path = repo / "labs/data/rag-corpus/pdf/easy/E02.pdf"


<h3>기본 추출과 표 추출 비교</h3>
<p><strong>확인할 점</strong> 같은 쪽을 두 파서로 읽고 글자 수와 표 개수를 함께 봅니다. 수치가 크다는 사실만으로 구조가 정확하다고 판단하지 않습니다.</p>

In [3]:
reader = PdfReader(pdf_path)
plain = reader.pages[1].extract_text() or ""
with pdfplumber.open(pdf_path) as pdf:
    layout = pdf.pages[1].extract_text(layout=True) or ""
    tables = pdf.pages[1].extract_tables()
print({"pages": len(reader.pages), "plain_chars": len(plain),
       "layout_chars": len(layout), "tables_on_page_2": len(tables)})


{'pages': 5, 'plain_chars': 884, 'layout_chars': 5311, 'tables_on_page_2': 1}


<h3>일부러 실패해 보기</h3>
<p><strong>실패 사례</strong> 스캔 PDF에는 텍스트 층이 없을 수 있습니다. OCR을 실행했다고 가장하지 않고 OCR이 필요한 쪽을 표시합니다.</p>

In [4]:
scan_path = repo / "labs/data/rag-corpus/pdf/scanned/S01-E01-scan.pdf"
if not scan_path.is_file(): raise FileNotFoundError(scan_path)
scan = PdfReader(scan_path)
missing = [i for i, p in enumerate(scan.pages, 1)
           if len((p.extract_text() or "").strip()) < 20]
assert missing == list(range(1, len(scan.pages) + 1))
print("OCR candidate pages:", missing)


OCR candidate pages: [1, 2, 3, 4, 5]
